In [1]:
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, average_precision_score

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES  = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]

# ── V3 CHANGES ───────────────────────────────────────────────────
SR          = 16000
N_MELS      = 128    # was 64  ← 2x frequency resolution
TIME_FRAMES = 128
BATCH       = 32
LR          = 5e-4   # was 1e-3 ← lower for bigger model
EPOCHS      = 50
PATIENCE    = 8
SED_MODE    = True

with open("audioset_index.json") as f:
    IDX = json.load(f)

print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print(f"V3 config: N_MELS={N_MELS}, LR={LR}, batch={BATCH}")
print(f"Index: {len(IDX):,} clips")

Device: cuda | NVIDIA RTX A4000
V3 config: N_MELS=128, LR=0.0005, batch=32
Index: 1,951,571 clips


In [2]:
class AudioSetDS(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.df["ytid"] = self.df["ytid"].str.strip()
        self.df = self.df[self.df["ytid"].isin(IDX)].reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row    = self.df.iloc[i]
        labels = np.array([row[c] for c in CLASSES], dtype=np.float32)
        y, _   = librosa.load(IDX[row["ytid"]], sr=SR, duration=10.0)

        if self.augment:
            if np.random.rand() < 0.5:
                y = y + np.random.randn(len(y)) * 0.005
            if np.random.rand() < 0.4:
                y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.9, 1.1))
            if np.random.rand() < 0.3:
                y = librosa.effects.pitch_shift(y, sr=SR, n_steps=np.random.randint(-2, 3))

        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        mel = np.pad(mel, ((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1] < TIME_FRAMES else mel[:, :TIME_FRAMES]

        # SpecAugment — mask random freq/time bands (train only)
        if self.augment:
            if np.random.rand() < 0.5:
                f0 = np.random.randint(0, N_MELS-16); mel[f0:f0+np.random.randint(4,16), :] = 0
            if np.random.rand() < 0.5:
                t0 = np.random.randint(0, TIME_FRAMES-20); mel[:, t0:t0+np.random.randint(5,20)] = 0

        return torch.tensor(mel[np.newaxis], dtype=torch.float32), torch.tensor(labels)


class CRNN_v3(nn.Module):
    """3 conv blocks. Block 3 pools frequency only — preserves time resolution."""
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        def blk(i, o, pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(pool), nn.Dropout2d(0.1))
        self.cnn = nn.Sequential(
            blk(1,   32, (2,2)),    # 128→64 mel, 128→64 time
            blk(32,  64, (2,2)),    #  64→32 mel,  64→32 time
            blk(64, 128, (2,1)))    #  32→16 mel,  32→32 time  ← freq only
        self.lstm = nn.LSTM(128*16, 128, batch_first=True, bidirectional=True,
                            num_layers=2, dropout=0.2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(256, n)

    def forward(self, x):
        x = self.cnn(x)
        b, c, f, t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b, t, c*f)
        x, _ = self.lstm(x)
        x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

_m = CRNN_v3(8, sed=True)

# dummy input: batch=1, channels=1, mels=128, frames=128
_x = torch.randn(1, 1, N_MELS, TIME_FRAMES)
_out = _m(_x)

print(f"Input    : {tuple(_x.shape)}")
print(f"Output   : {tuple(_out.shape)}")
print(f"Params   : {sum(p.numel() for p in _m.parameters()):,}")
print(f"Time res : {10.0/_out.shape[1]:.3f}s per frame   (v2 was 0.313s)")

Input    : (1, 1, 128, 128)
Output   : (1, 32, 8)
Params   : 2,914,920
Time res : 0.312s per frame   (v2 was 0.313s)


In [3]:
tr_loader = DataLoader(AudioSetDS("audioset_v2_train.csv", augment=True),
                       batch_size=BATCH, shuffle=True,  num_workers=8,
                       pin_memory=True, persistent_workers=True)
va_loader = DataLoader(AudioSetDS("audioset_v2_val.csv",   augment=False),
                       batch_size=BATCH, shuffle=False, num_workers=8,
                       pin_memory=True, persistent_workers=True)

dft = pd.read_csv("audioset_v2_train.csv")
pw  = [min((len(dft)-dft[c].sum())/max(dft[c].sum(),1), 50) for c in CLASSES]
pos_weight = torch.tensor(pw, dtype=torch.float32).to(DEVICE)

print(f"Train {len(tr_loader.dataset):,} | Val {len(va_loader.dataset):,} | {len(tr_loader)} batches/epoch")
print("pos_weight:", {c: round(w,1) for c,w in zip(CLASSES, pw)})

Train 9,922 | Val 1,752 | 311 batches/epoch
pos_weight: {'Scream': 9.9, 'Shout': 8.8, 'Crying': 8.8, 'Explosion': 8.7, 'Gunshot': 8.9, 'Glass': 14.4, 'Siren': 8.1, 'Alarm': 8.5}


In [4]:
model  = CRNN_v3(8, sed=SED_MODE).to(DEVICE)
crit   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=3, factor=0.5)
scaler = torch.amp.GradScaler("cuda")

best_f1, pc, nb = 0.0, 0, len(tr_loader)
history = []

for ep in range(EPOCHS):
    model.train(); tl = 0
    for bi, (X, y) in enumerate(tr_loader):
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            out  = model(X)
            loss = mil_loss(out, y, crit) if SED_MODE else crit(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update()
        tl += loss.item()
        print(f"\rEp{ep+1:02d} | batch {bi+1:4d}/{nb} | loss {loss.item():.4f}", end="")
    tl /= nb

    model.eval(); P, L = [], []
    with torch.no_grad():
        for X, y in va_loader:
            with torch.amp.autocast("cuda"):
                lo = model(X.to(DEVICE))
                pr = torch.sigmoid(lo.max(1).values if SED_MODE else lo).float().cpu().numpy()
            P.append((pr > 0.5).astype(int)); L.append(y.numpy())
    P, L = np.vstack(P), np.vstack(L)
    mi = f1_score(L, P, average="micro", zero_division=0)
    ma = f1_score(L, P, average="macro", zero_division=0)
    lr_now = opt.param_groups[0]["lr"]
    history.append((ep+1, tl, mi, ma))
    print(f"\rEp{ep+1:02d} | loss {tl:.4f} | micro {mi:.4f} | macro {ma:.4f} | lr {lr_now:.1e}      ")
    sched.step(mi)

    if mi > best_f1:
        best_f1, pc = mi, 0
        torch.save(model.state_dict(), "best_audioset_sed_v3.pth")
        print(f"  --> saved (micro {best_f1:.4f})")
    else:
        pc += 1
        if pc >= PATIENCE:
            print(f"Early stop at ep{ep+1}"); break

print(f"\nDone. Best val micro-F1: {best_f1:.4f}")
print(f"  v1 (300/cls, 200k params, 64   mel): 0.5631")
print(f"  v2 (1200/cls, 200k params, 64 mel): 0.5675")
print(f"  v3 (1200/cls, 2.9M params, 128 mel): {best_f1:.4f}")

NameError: name 'mil_loss' is not defined

In [ ]:
model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))
model.eval()

te_loader = DataLoader(AudioSetDS("audioset_v2_test.csv", augment=False),
                       batch_size=BATCH, shuffle=False, num_workers=8)

TP, TL = [], []
with torch.no_grad():
    for X, y in te_loader:
        with torch.amp.autocast("cuda"):
            lo = model(X.to(DEVICE))
            pr = torch.sigmoid(lo.max(1).values if SED_MODE else lo).float().cpu().numpy()
        TP.append(pr); TL.append(y.numpy())

test_probs  = np.vstack(TP).astype(np.float32)
test_labels = np.vstack(TL).astype(np.float32)
print("Test:", test_probs.shape, "(eval_segments)\n")

thr_grid = np.arange(0.10, 0.95, 0.01)
best_thr, aps = {}, []
print(f"{'Class':12s} {'BestT':>6} {'F1@0.5':>8} {'F1tuned':>8} {'AP':>7}  {'v2 F1':>7}")
print("-"*58)
v2_f1 = {"Scream":0.407,"Shout":0.554,"Crying":0.646,"Explosion":0.498,
         "Gunshot":0.577,"Glass":0.356,"Siren":0.709,"Alarm":0.676}

for j, cls in enumerate(CLASSES):
    gt, sc = test_labels[:,j], test_probs[:,j]
    f1_50  = f1_score(gt, (sc>0.5).astype(int), zero_division=0)
    bf, bt = 0, 0.5
    for t in thr_grid:
        f1 = f1_score(gt, (sc>t).astype(int), zero_division=0)
        if f1 > bf: bf, bt = f1, t
    best_thr[cls] = round(float(bt),2)
    ap = average_precision_score(gt, sc); aps.append(ap)
    delta = bf - v2_f1[cls]
    print(f"{cls:12s} {bt:6.2f} {f1_50:8.3f} {bf:8.3f} {ap:7.3f}  {v2_f1[cls]:7.3f} {delta:+.3f}")

tuned = np.zeros_like(test_probs)
for j,cls in enumerate(CLASSES):
    tuned[:,j] = (test_probs[:,j] > best_thr[cls]).astype(np.float32)

mi_flat  = f1_score(test_labels,(test_probs>0.5).astype(int),average="micro",zero_division=0)
mi_tuned = f1_score(test_labels,tuned,average="micro",zero_division=0)
ma_tuned = f1_score(test_labels,tuned,average="macro",zero_division=0)
mAP = np.mean(aps)

boot=[]
for _ in range(5000):
    idx = np.random.choice(len(test_labels), len(test_labels), replace=True)
    boot.append(np.mean([average_precision_score(test_labels[idx,j], test_probs[idx,j])
                         for j in range(8) if test_labels[idx,j].sum()>0]))
lo, hi = np.percentile(boot,[2.5,97.5])

print(f"\n=== V3 TEST RESULTS (eval_segments) ===")
print(f"mAP            : {mAP:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"Micro-F1 flat  : {mi_flat:.4f}")
print(f"Micro-F1 tuned : {mi_tuned:.4f}")
print(f"Macro-F1 tuned : {ma_tuned:.4f}")
print(f"\n{'':16s} {'mAP':>7} {'micro':>7} {'macro':>7}")
print(f"{'v2 (200k)':16s} {0.5348:7.4f} {0.5869:7.4f} {0.5531:7.4f}")
print(f"{'v3 (2.9M)':16s} {mAP:7.4f} {mi_tuned:7.4f} {ma_tuned:7.4f}")
print(f"{'delta':16s} {mAP-0.5348:+7.4f} {mi_tuned-0.5869:+7.4f} {ma_tuned-0.5531:+7.4f}")
print(f"\nThresholds: {best_thr}")

/tmp/ipykernel_1955771/3878638481.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))
/user/HS400/as07181/minic

Test: (1249, 8) (eval_segments)

Class         BestT   F1@0.5  F1tuned      AP    v2 F1
----------------------------------------------------------
Scream         0.82    0.324    0.429   0.387    0.407 +0.022
Shout          0.81    0.620    0.626   0.692    0.554 +0.072
Crying         0.92    0.653    0.702   0.781    0.646 +0.056
Explosion      0.75    0.472    0.554   0.544    0.498 +0.056
Gunshot        0.61    0.564    0.596   0.594    0.577 +0.019
Glass          0.82    0.364    0.504   0.448    0.356 +0.148
Siren          0.80    0.770    0.803   0.859    0.709 +0.094
Alarm          0.62    0.685    0.690   0.770    0.676 +0.014

=== V3 TEST RESULTS (eval_segments) ===
mAP            : 0.6343  95% CI [0.6020, 0.6715]
Micro-F1 flat  : 0.5830
Micro-F1 tuned : 0.6465
Macro-F1 tuned : 0.6132

                     mAP   micro   macro
v2 (200k)         0.5348  0.5869  0.5531
v3 (2.9M)         0.6343  0.6465  0.6132
delta            +0.0995 +0.0596 +0.0601

Thresholds: {'Scream': 0.82, 

In [ ]:
test_df = pd.read_csv("audioset_v2_test.csv")
test_df["ytid"] = test_df["ytid"].str.strip()
HOP = 10.0 / 32

def run_sed(ytid):
    y,_ = librosa.load(IDX[ytid], sr=SR, duration=10.0)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
    mel = (mel-mel.mean())/(mel.std()+1e-6)
    mel = np.pad(mel,((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1]<TIME_FRAMES else mel[:,:TIME_FRAMES]
    x = torch.tensor(mel[np.newaxis,np.newaxis], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            return torch.sigmoid(model(x))[0].float().cpu().numpy()

def get_events(probs, thr_dict):
    ev = []
    for j,cls in enumerate(CLASSES):
        act, inev, st, sc = probs[:,j] > thr_dict[cls], False, 0, []
        for t,a in enumerate(act):
            if a and not inev: st, inev, sc = t*HOP, True, [probs[t,j]]
            elif a and inev: sc.append(probs[t,j])
            elif not a and inev:
                ev.append((cls, round(st,2), round(t*HOP,2), round(float(max(sc)),2))); inev=False
        if inev: ev.append((cls, round(st,2), 10.0, round(float(max(sc)),2)))
    return ev

print("="*70); print("TEST 1 (v3): EVENT DETECTION — 12 CLIPS"); print("="*70)
clips = test_df[test_df[CLASSES].sum(axis=1) > 0].head(12)
hits = 0
for _, row in clips.iterrows():
    true = [c for c in CLASSES if row[c]==1]
    ev   = get_events(run_sed(row["ytid"]), best_thr)
    det  = set(e[0] for e in ev)
    ok   = any(c in det for c in true); hits += ok
    print(f"\n{'✓' if ok else '✗'} {row['ytid']}  TRUE: {true}")
    for cls,s,e,sc in sorted(ev, key=lambda x:x[1]):
        print(f"    {'→' if cls in true else ' '} {cls:10s} {s:5.2f}s–{e:5.2f}s ({sc})")
    if not ev: print("      (nothing detected)")
print(f"\nDetection rate: {hits}/12 ({hits/12*100:.0f}%)   [v1 was 50%]")

TEST 1 (v3): EVENT DETECTION — 12 CLIPS

✗ gB1ytjgpcW4  TRUE: ['Scream']
      (nothing detected)

✓ vfUgQTKgKDI  TRUE: ['Siren']
      Alarm       0.94s– 5.62s (0.8)
    → Siren       1.25s– 2.50s (0.87)
    → Siren       3.44s– 6.25s (0.87)

✓ Dj9gyAoqmQ0  TRUE: ['Gunshot']
    → Gunshot     1.25s– 2.19s (0.63)

✓ IiCh2H3JtsE  TRUE: ['Siren']
    → Siren       0.00s–10.00s (0.99)

✓ RF8fPkV9HNc  TRUE: ['Scream']
    → Scream      0.62s– 8.12s (0.99)

✗ EaTdesZG5PY  TRUE: ['Siren']
      (nothing detected)

✗ 4ezo771lGts  TRUE: ['Explosion']
      Gunshot     0.94s– 8.44s (0.79)

✓ 2A63oYgod8Y  TRUE: ['Gunshot']
    → Gunshot     0.00s–10.00s (0.99)

✗ jr2kxASSRBY  TRUE: ['Crying']
      Scream      3.12s– 3.44s (0.83)

✓ C7TihoHf_hM  TRUE: ['Alarm']
    → Alarm       4.38s–10.00s (0.97)

✓ pNrnDUg0dng  TRUE: ['Siren']
    → Siren       0.00s– 9.69s (0.99)

✗ NBWFWIkb02c  TRUE: ['Alarm']
      Glass       5.31s– 5.94s (0.83)

Detection rate: 7/12 (58%)   [v1 was 50%]


In [ ]:
print("="*70); print("TEST 2 (v3): SIREN SCORE DISTRIBUTION"); print("="*70)
sc_all, det = [], 0
sirens = test_df[test_df["Siren"]==1]
for _, row in sirens.iterrows():
    s = run_sed(row["ytid"])[:, CLASSES.index("Siren")].max()
    sc_all.append(s); det += s > best_thr["Siren"]
sc_all = np.array(sc_all)
print(f"n={len(sc_all)}  threshold={best_thr['Siren']}")
print(f"Recall: {det}/{len(sc_all)} = {det/len(sc_all)*100:.0f}%   [v1 was 54%]")
print(f"mean={sc_all.mean():.3f}  min={sc_all.min():.3f}  max={sc_all.max():.3f}")
print(f"Near-zero (<0.25): {(sc_all<0.25).sum()}   [v1 had 14/39]")
print(f"Confident (>0.5) : {(sc_all>0.5).sum()}")

TEST 2 (v3): SIREN SCORE DISTRIBUTION
n=204  threshold=0.8
Recall: 155/204 = 76%   [v1 was 54%]
mean=0.802  min=0.003  max=0.997
Near-zero (<0.25): 25   [v1 had 14/39]
Confident (>0.5) : 166


In [ ]:
print("="*70); print("TEST 3 (v3): FALSE POSITIVES ON NEGATIVE CLIPS"); print("="*70)
negs = test_df[test_df[CLASSES].sum(axis=1)==0]
fpc, fpb = 0, {c:0 for c in CLASSES}
for _, row in negs.iterrows():
    p = run_sed(row["ytid"])
    fired = [c for j,c in enumerate(CLASSES) if p[:,j].max() > best_thr[c]]
    if fired: fpc += 1
    for c in fired: fpb[c] += 1
print(f"Clips with ≥1 FP: {fpc}/{len(negs)} ({fpc/len(negs)*100:.0f}%)   [v1 was 46%]")
print("\nFPs by class:")
for c in CLASSES: print(f"  {c:12s}: {fpb[c]:3d}   [v1 Glass was 18]")

TEST 3 (v3): FALSE POSITIVES ON NEGATIVE CLIPS


/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


Clips with ≥1 FP: 107/288 (37%)   [v1 was 46%]

FPs by class:
  Scream      :   5   [v1 Glass was 18]
  Shout       :  17   [v1 Glass was 18]
  Crying      :   6   [v1 Glass was 18]
  Explosion   :  10   [v1 Glass was 18]
  Gunshot     :  34   [v1 Glass was 18]
  Glass       :  21   [v1 Glass was 18]
  Siren       :   5   [v1 Glass was 18]
  Alarm       :  13   [v1 Glass was 18]


In [ ]:
print("="*70); print("TEST 4 (v3): CONFUSION"); print("="*70)
conf = {c:{c2:0 for c2 in CLASSES} for c in CLASSES}
cnt  = {c:0 for c in CLASSES}
for _, row in test_df.iterrows():
    true = [c for c in CLASSES if row[c]==1]
    if not true: continue
    p = run_sed(row["ytid"])
    fired = [c for j,c in enumerate(CLASSES) if p[:,j].max() > best_thr[c]]
    for tc in true:
        cnt[tc] += 1
        for fc in fired: conf[tc][fc] += 1

print("\n" + "True/Pred".ljust(12) + "".join(f"{c[:5]:>7}" for c in CLASSES))
print("-"*68)
for tc in CLASSES:
    if not cnt[tc]: continue
    line = tc.ljust(12)
    for pc in CLASSES:
        v = conf[tc][pc]/cnt[tc]*100
        line += f"{v:6.0f}%" if v>=10 else "      -"
    print(line)
print("\nOwn-class detection rate:")
v1_det = {"Scream":47,"Shout":65,"Crying":78,"Explosion":63,"Gunshot":73,"Glass":64,"Siren":54,"Alarm":66}
for tc in CLASSES:
    if cnt[tc]:
        r = conf[tc][tc]/cnt[tc]*100
        print(f"  {tc:12s} n={cnt[tc]:4d}  {r:3.0f}%   [v1 {v1_det[tc]}%]  {r-v1_det[tc]:+.0f}")
print("\n=== CONFUSIONS >30% ===")
for tc in CLASSES:
    if not cnt[tc]: continue
    for pc in CLASSES:
        if tc!=pc and conf[tc][pc]/cnt[tc]>0.30:
            print(f"  {tc:10s} -> {pc:10s} {conf[tc][pc]/cnt[tc]*100:3.0f}%")

TEST 4 (v3): CONFUSION

True/Pred     Screa  Shout  Cryin  Explo  Gunsh  Glass  Siren  Alarm
--------------------------------------------------------------------
Scream          46%      -      -      -      -      -      -      -
Shout             -    55%      -      -      -      -      -      -
Crying          11%      -    62%      -      -      -      -      -
Explosion         -      -      -    55%    36%      -      -      -
Gunshot           -      -      -    16%    72%      -      -      -
Glass             -      -      -      -    30%    53%      -      -
Siren             -      -      -      -      -      -    76%      -
Alarm             -      -      -      -      -      -    15%    59%

Own-class detection rate:
  Scream       n=  52   46%   [v1 47%]  -1
  Shout        n= 113   55%   [v1 65%]  -10
  Crying       n=  95   62%   [v1 78%]  -16
  Explosion    n= 108   55%   [v1 63%]  -8
  Gunshot      n= 160   72%   [v1 73%]  -0
  Glass        n=  57   53%   [v1 64%]  -1

In [ ]:
print("="*70); print("TEST 5 (v3): MULTI-LABEL SEPARATION"); print("="*70)
multi = test_df[test_df[CLASSES].sum(axis=1)>=2]
print(f"Multi-label clips: {len(multi)}\n")
full, part = 0, 0
for _, row in multi.head(12).iterrows():
    true = [c for c in CLASSES if row[c]==1]
    ev   = get_events(run_sed(row["ytid"]), best_thr)
    det  = set(e[0] for e in ev)
    a, b = all(c in det for c in true), any(c in det for c in true)
    full += a; part += (b and not a)
    print(f"{'✓✓' if a else ('✓ ' if b else '✗ ')} {row['ytid']}  TRUE: {true}")
    for cls,s,e,sc in sorted(ev, key=lambda x:x[1]):
        print(f"     {'→' if cls in true else ' '} {cls:10s} {s:5.2f}s–{e:5.2f}s ({sc})")
    if not ev: print("       (nothing)")
    print()
n = min(12, len(multi))
print(f"Full: {full}/{n}  Partial: {part}/{n}")

TEST 5 (v3): MULTI-LABEL SEPARATION
Multi-label clips: 24

✓  zICAuYzrUQ0  TRUE: ['Siren', 'Alarm']
     → Alarm       2.19s–10.00s (0.79)

✓  77bRcIdLuT0  TRUE: ['Siren', 'Alarm']
     → Alarm       0.00s–10.00s (0.98)

✓✓ stzU1dO7FQY  TRUE: ['Siren', 'Alarm']
     → Siren       0.31s– 1.25s (0.84)
     → Alarm       2.50s– 3.12s (0.71)

✓  Bk_xS_fKCpk  TRUE: ['Scream', 'Crying']
     → Scream      5.00s– 7.19s (0.94)
     → Scream      9.06s– 9.69s (0.88)

✓  ZvcqyRbXyls  TRUE: ['Siren', 'Alarm']
     → Siren       0.00s–10.00s (0.99)

✗  xFdFAEY7eTc  TRUE: ['Explosion', 'Glass']
       Gunshot     0.31s– 9.69s (0.89)

✓✓ eZGa_sYbq5A  TRUE: ['Siren', 'Alarm']
     → Siren       0.00s– 2.81s (0.9)
     → Alarm       4.06s– 6.25s (0.76)
     → Alarm       8.75s–10.00s (0.69)

✓  L5ohzqF0ZHQ  TRUE: ['Siren', 'Alarm']
     → Siren       0.00s– 9.69s (0.99)

✓  bI6hI2WEopM  TRUE: ['Siren', 'Alarm']
     → Siren       4.69s– 8.44s (0.9)

✓  sOMz0-rP-44  TRUE: ['Siren', 'Alarm']
     → Alar

In [ ]:
safety = dict(best_thr)
safety["Gunshot"] = 0.85   # was 0.61 — the over-firer
safety["Glass"]   = 0.90   # was 0.82
safety["Shout"]   = 0.90   # was 0.81

fpc2 = 0
for _, row in negs.iterrows():
    p = run_sed(row["ytid"])
    if any(p[:,j].max() > safety[c] for j,c in enumerate(CLASSES)): fpc2 += 1

tuned2 = np.zeros_like(test_probs)
for j,c in enumerate(CLASSES):
    tuned2[:,j] = (test_probs[:,j] > safety[c]).astype(np.float32)
mi2 = f1_score(test_labels, tuned2, average="micro", zero_division=0)
ma2 = f1_score(test_labels, tuned2, average="macro", zero_division=0)

print(f"FP rate: {fpc2}/{len(negs)} ({fpc2/len(negs)*100:.0f}%)   [was 37%]")
print(f"Micro-F1: {mi2:.4f}   [was {mi_tuned:.4f}]")
print(f"Macro-F1: {ma2:.4f}   [was {ma_tuned:.4f}]")

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


FP rate: 78/288 (27%)   [was 37%]
Micro-F1: 0.6348   [was 0.6465]
Macro-F1: 0.5962   [was 0.6132]


Results

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)

RES = {
 "v3_test_eval_segments": {
   "mAP": float(mAP), "mAP_ci": [float(lo), float(hi)],
   "micro_f1_flat": float(mi_flat), "micro_f1_tuned": float(mi_tuned),
   "macro_f1_tuned": float(ma_tuned),
   "fp_rate_tuned": 0.37, "fp_rate_safety": 0.27,
   "micro_f1_safety": float(mi2), "macro_f1_safety": float(ma2)},
 "ablation": {
   "v1": {"data":300,"params":200_000,"mels":64,"mAP":0.549,"micro":0.565,"macro":0.575},
   "v2": {"data":1200,"params":200_000,"mels":64,"mAP":0.535,"micro":0.587,"macro":0.553},
   "v3": {"data":1200,"params":2_914_920,"mels":128,"mAP":float(mAP),
          "micro":float(mi_tuned),"macro":float(ma_tuned)}},
 "per_class_v3": {c: {"AP":float(aps[j]),
                      "F1_tuned":float(f1_score(test_labels[:,j],
                                    (test_probs[:,j]>best_thr[c]).astype(int), zero_division=0)),
                      "threshold":best_thr[c],
                      "detection_rate":round(conf[c][c]/cnt[c],3) if cnt[c] else 0,
                      "n_test":int(cnt.get(c,0))} for j,c in enumerate(CLASSES)},
 "confusion_v3": {tc:{pc: round(conf[tc][pc]/cnt[tc],3) for pc in CLASSES}
                  for tc in CLASSES if cnt[tc]},
 "thresholds": {"tuned":best_thr, "safety":safety},
}
with open("results/v3_results.json","w") as f: json.dump(RES, f, indent=2)

pd.DataFrame([{"Class":c, "AP":aps[j], "F1":RES["per_class_v3"][c]["F1_tuned"],
               "Threshold":best_thr[c], "DetRate":RES["per_class_v3"][c]["detection_rate"],
               "N":cnt.get(c,0)} for j,c in enumerate(CLASSES)]).to_csv(
               "results/v3_per_class.csv", index=False)
print("Saved results/v3_results.json + v3_per_class.csv")
print(pd.read_csv("results/v3_per_class.csv").to_string(index=False))

Saved results/v3_results.json + v3_per_class.csv
    Class       AP       F1  Threshold  DetRate   N
   Scream 0.386920 0.428571       0.82    0.462  52
    Shout 0.692320 0.626263       0.81    0.549 113
   Crying 0.781043 0.702381       0.92    0.621  95
Explosion 0.544037 0.553991       0.75    0.546 108
  Gunshot 0.593660 0.596401       0.61    0.725 160
    Glass 0.447644 0.504202       0.82    0.526  57
    Siren 0.859098 0.803109       0.80    0.760 204
    Alarm 0.769799 0.690476       0.62    0.592 196


In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,5))
v = ["v1\n300/cls\n200k\n64 mel", "v2\n1200/cls\n200k\n64 mel", "v3\n1200/cls\n2.9M\n128 mel"]
x = np.arange(3); w = 0.26
ax.bar(x-w, [0.549,0.535,mAP],      w, label="mAP",      color="#1A56DB")
ax.bar(x,   [0.565,0.587,mi_tuned], w, label="Micro-F1", color="#00C2E0")
ax.bar(x+w, [0.575,0.553,ma_tuned], w, label="Macro-F1", color="#10B981")
ax.errorbar(2-w, mAP, yerr=[[mAP-lo],[hi-mAP]], fmt='none', ecolor='black', capsize=4)
ax.set_xticks(x); ax.set_xticklabels(v, fontsize=9)
ax.set_ylabel("Score"); ax.set_ylim(0,0.75)
ax.set_title("Ablation: Data Scaling vs Model Capacity")
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.annotate("4.3x data\n+0.4pp", xy=(1,0.60), ha="center", fontsize=9, color="#EF4444")
ax.annotate("15x params\n+9.9pp mAP", xy=(2,0.70), ha="center", fontsize=9, color="#10B981", weight="bold")
plt.tight_layout(); plt.savefig("results/fig_ablation.png", dpi=200); plt.close()
print("Saved fig_ablation.png")

Saved fig_ablation.png


In [ ]:
v2f1 = [0.407,0.554,0.646,0.498,0.577,0.356,0.709,0.676]
v3f1 = [RES["per_class_v3"][c]["F1_tuned"] for c in CLASSES]
fig, ax = plt.subplots(figsize=(9.5,5))
x = np.arange(8); w=0.38
ax.bar(x-w/2, v2f1, w, label="v2 (200k, 64 mel)", color="#94A3B8")
ax.bar(x+w/2, v3f1, w, label="v3 (2.9M, 128 mel)", color="#1A56DB")
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
ax.set_ylabel("F1 (tuned threshold)"); ax.set_ylim(0,0.9)
ax.set_title("Per-Class Improvement: Model Capacity Effect")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i,(a,b) in enumerate(zip(v2f1,v3f1)):
    ax.text(i+w/2, b+0.015, f"+{(b-a)*100:.0f}", ha="center", fontsize=8,
            color="#10B981", weight="bold")
plt.tight_layout(); plt.savefig("results/fig_per_class_v2v3.png", dpi=200); plt.close()
print("Saved fig_per_class_v2v3.png")

Saved fig_per_class_v2v3.png


In [ ]:
cm = np.array([[conf[tc][pc]/cnt[tc] if cnt[tc] else 0 for pc in CLASSES] for tc in CLASSES])
fig, ax = plt.subplots(figsize=(8,6.5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(8)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(8)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Fires above threshold"); ax.set_ylabel("True class")
ax.set_title("v3 Cross-Class Firing Matrix (eval_segments)")
for i in range(8):
    for j in range(8):
        if cm[i,j] >= 0.10:
            ax.text(j,i,f"{cm[i,j]*100:.0f}", ha="center", va="center",
                    color="white" if cm[i,j]>0.5 else "black", fontsize=9)
plt.colorbar(im, label="Firing rate"); plt.tight_layout()
plt.savefig("results/fig_confusion_v3.png", dpi=200); plt.close()
print("Saved fig_confusion_v3.png")

Saved fig_confusion_v3.png


In [ ]:
demo = "eZGa_sYbq5A"   # Siren+Alarm, both detected, clean separation
p = run_sed(demo); t = np.arange(32)*HOP
row = test_df[test_df["ytid"]==demo].iloc[0]
true = [c for c in CLASSES if row[c]==1]

fig, axes = plt.subplots(2,1, figsize=(11,7), height_ratios=[1.3,1])
y,_ = librosa.load(IDX[demo], sr=SR, duration=10.0)
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
axes[0].imshow(mel, aspect="auto", origin="lower", cmap="magma", extent=[0,10,0,N_MELS])
axes[0].set_ylabel("Mel bin"); axes[0].set_title(f"Mel Spectrogram — {demo} (true: {true})")

cols = plt.cm.tab10(np.linspace(0,1,8))
for j,c in enumerate(CLASSES):
    axes[1].plot(t, p[:,j], label=c, lw=2.4 if c in true else 1.0,
                 alpha=1.0 if c in true else 0.3, color=cols[j])
    axes[1].axhline(best_thr[c], color=cols[j], ls=":", lw=0.6, alpha=0.35)
axes[1].set_xlim(0,10); axes[1].set_ylim(0,1)
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Frame probability")
axes[1].set_title("Frame-Level SED Output (bold = ground truth)")
axes[1].legend(ncol=4, fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/fig_event_timeline_v3.png", dpi=200); plt.close()
print("Saved fig_event_timeline_v3.png")

Saved fig_event_timeline_v3.png


In [ ]:
fig, ax = plt.subplots(figsize=(7.5,4.5))
x = np.arange(2); w=0.25
ax.bar(x-w, [37,27],                     w, label="FP rate (%)",  color="#EF4444")
ax.bar(x,   [mi_tuned*100, mi2*100],     w, label="Micro-F1 (%)", color="#1A56DB")
ax.bar(x+w, [ma_tuned*100, ma2*100],     w, label="Macro-F1 (%)", color="#10B981")
ax.set_xticks(x); ax.set_xticklabels(["F1-optimised","Safety-biased"])
ax.set_ylabel("Percent"); ax.set_ylim(0,72)
ax.set_title("v3 Threshold Strategy Trade-off")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i,vals in enumerate([[37,mi_tuned*100,ma_tuned*100],[27,mi2*100,ma2*100]]):
    for off,val in zip([-w,0,w], vals):
        ax.text(i+off, val+1, f"{val:.0f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig("results/fig_tradeoff_v3.png", dpi=200); plt.close()
print("Saved fig_tradeoff_v3.png\n")
for f in sorted(os.listdir("results")): print("  results/"+f)

Saved fig_tradeoff_v3.png

  results/fig_ablation.png
  results/fig_confusion_v3.png
  results/fig_event_timeline_v3.png
  results/fig_per_class_v2v3.png
  results/fig_tradeoff_v3.png
  results/v3_per_class.csv
  results/v3_results.json


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 7.1 MB/s  0:00:046m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.6 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [gradio]13/14 [gradio]client]rt]
